# Arm comparison at common checkpoints
Run this after both arms have trained. It picks checkpoints both arms reached, evaluates each arm there on the twenty fixed holdout scenarios, and reports paired per-seed differences.

The headline is the fifteen primary-test seeds that model selection never saw. Arms are never ranked by `best_model.sb3`: its selection averages five scenarios and its evaluation-to-evaluation standard deviation was 0.16 in the v7 run.

Comparing several checkpoints in one run answers whether a gap holds throughout training or only appears late. The slow step is reading every checkpoint timestep, and it runs once for the whole sweep.

This notebook tracks `main` on purpose. It is an analysis tool, not part of an immutable training release, so it is not pinned to a tag.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

# Edit these if the arms live elsewhere.
RAW_DIR = Path("/content/drive/MyDrive/CNN-RL-improved/raw-direct-830k-seed0/ppo")
CNN_DIR = Path("/content/drive/MyDrive/CNN-RL-improved/scale-aware-cnn-6h-seed0/ppo")
OUTPUT_DIR = Path("/content/drive/MyDrive/CNN-RL-improved/comparison-830k")

# Checkpoints to compare. Empty list = the largest checkpoint both arms reached.
TIMESTEPS = [200_000, 400_000, 600_000, 830_000]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for label, path in (("raw_direct", RAW_DIR), ("candidate_cnn", CNN_DIR)):
    checkpoints = sorted((path / "checkpoints").glob("*.sb3")) if (path / "checkpoints").is_dir() else []
    if not checkpoints:
        raise RuntimeError(f"{label} has no checkpoints under {path}")
    print(f"{label}: {len(checkpoints)} checkpoints, newest {checkpoints[-1].name}")


In [ ]:
import subprocess
REPOSITORY = "https://github.com/LMS4681/CNN-RL-Raw-Comparison.git"
CHECKOUT = Path("/content/compare")
# A separate checkout: the training notebooks pin immutable tags that predate
# comparison/arm_comparison.py, so this analysis tracks main instead.
if (CHECKOUT / ".git").is_dir():
    subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(CHECKOUT), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(CHECKOUT)], check=True)
ALLOC_RL = CHECKOUT / "AllocRL"
head = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
print("comparison checkout:", head)


In [ ]:
import subprocess, sys
# Same locked dependency set the training runs used, so archives load identically.
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--require-hashes",
                "-r", str(ALLOC_RL / "requirements-comparison.txt")], check=True)


In [ ]:
import os, subprocess, sys
os.chdir(ALLOC_RL)
command = [sys.executable, "-u", "-m", "comparison.arm_comparison",
           "--raw-dir", str(RAW_DIR), "--cnn-dir", str(CNN_DIR),
           "--scenarios", "./data/fixed_eval_scenarios.json",
           "--output-dir", str(OUTPUT_DIR)]
for step in TIMESTEPS:
    command += ["--timestep", str(step)]
print(" ".join(command), "\n" + "=" * 70)
# The checkpoint inventory is read once for the whole sweep; each requested
# timestep then costs forty 913-decision episodes on CPU.
subprocess.run(command, check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
import json
summaries = {}
for path in sorted(OUTPUT_DIR.glob("arm_comparison_summary_*.json")):
    summary = json.loads(path.read_text(encoding="utf-8"))
    summaries[summary["common_timestep"]] = summary
if not summaries:
    raise RuntimeError(f"no comparison summaries under {OUTPUT_DIR}")

for timestep in sorted(summaries):
    summary = summaries[timestep]
    headline = summary["partitions"]["primary_test"]
    files = "  ".join(f"{arm}={reference['file']}" for arm, reference in summary["checkpoints"].items())
    print(f"\n=== timestep {timestep}  ({files}) ===")
    for metric, values in headline["paired"].items():
        low, high = values["bootstrap_ci_95"]
        direction = "lower is better" if values["lower_is_better"] else "higher is better"
        print(f"  {metric:<20} raw {headline['arms']['raw_direct'][metric]:+.4f}"
              f"  cnn {headline['arms']['candidate_cnn'][metric]:+.4f}"
              f"  diff {values['mean_difference']:+.4f}"
              f"  CI [{low:+.4f}, {high:+.4f}]"
              f"  {'excludes 0' if values['excludes_zero'] else 'includes 0'}"
              f"  cnn better {values['seeds_favouring_candidate']}/{values['seed_count']}  ({direction})")

print("\n=== terminal score trend (candidate_cnn minus raw_direct) ===")
for timestep in sorted(summaries):
    values = summaries[timestep]["partitions"]["primary_test"]["paired"]["mean_terminal_score"]
    low, high = values["bootstrap_ci_95"]
    print(f"  {timestep:>8}  diff {values['mean_difference']:+.4f}  CI [{low:+.4f}, {high:+.4f}]"
          f"  cnn better {values['seeds_favouring_candidate']:>2}/{values['seed_count']}")
print("\nartifacts:", sorted(p.name for p in OUTPUT_DIR.glob("arm_comparison*")))
